In [4]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION — unchanged from your pasted version
# -----------------------------------------------------------------------------
F_STOP_HZ = 50e6
CHIRP_SPAN_HZ = 350e6
F_START_HZ = F_STOP_HZ - CHIRP_SPAN_HZ

NUM_STEPS = 124
AMPLITUDE = 0.9

BUFFER_DURATION_S_REQUESTED = 6e-3
max_feasible_buffer_s = 0.95 * ENV_MAXLEN / ENV_SR
BUFFER_DURATION_S = min(BUFFER_DURATION_S_REQUESTED, max_feasible_buffer_s)

TOTAL_SWEEP_S = 6e-3
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

buffer_step_s = BUFFER_DURATION_S / NUM_STEPS
print(f"Each step's stored buffer: {buffer_step_s*1e9:.1f} ns of raw samples, "
      f"repeated via mode='periodic' for {STEP_HOLD_US:.2f} us before switching.")

freqs_hz = np.linspace(F_START_HZ, F_STOP_HZ, NUM_STEPS)

maxv = soccfg.get_maxv(GEN_CH)
idata_list = []
qdata_list = []

phase = 0.0
for f in freqs_hz:
    y, phase, n_samples = serrodyne_tone(f, buffer_step_s, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
    pad = (-len(y)) % samps_per_clk
    if pad:
        y = np.concatenate([y, np.zeros(pad)])
    i_wave = np.round(y * maxv).astype(np.int16)
    q_wave = np.zeros_like(i_wave)
    idata_list.append(i_wave)
    qdata_list.append(q_wave)

samples_per_step = len(idata_list[0])
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step buffer length: {samples_per_step} samples")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER: 4 hyperfine tones at the FINAL chirp frequency, concatenated
#     into ONE envelope. Played with mode="periodic", the hardware repeats
#     this whole buffer forever with no further tProc instructions at all --
#     no loop, no condj, nothing to interrupt.
# -----------------------------------------------------------------------------
BASE_TONES_HZ = np.array([-50e6, 0e6, 50e6, 100e6])   # the 4 CaF hyperfine offsets
NUM_TONES = len(BASE_TONES_HZ)

trap_i_pieces = []
trap_q_pieces = []
for base_tone in BASE_TONES_HZ:
    f = F_STOP_HZ + base_tone   # centered on where the sweep ends
    y, phase, n_samples = serrodyne_tone(f, buffer_step_s, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
    pad = (-len(y)) % samps_per_clk
    if pad:
        y = np.concatenate([y, np.zeros(pad)])
    trap_i_pieces.append(np.round(y * maxv).astype(np.int16))
    trap_q_pieces.append(np.zeros(len(y), dtype=np.int16))

trap_idata = np.concatenate(trap_i_pieces)
trap_qdata = np.concatenate(trap_q_pieces)
print(f"Trap buffer: {len(trap_idata)} samples ({NUM_TONES} tones x {samples_per_step} samples each)")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or buffer size."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
prog.run(soc)
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Each step's stored buffer: 51.1 ns of raw samples, repeated via mode='periodic' for 48.39 us before switching.
Per-step buffer length: 512 samples
Total envelope samples (sweep): 63488 / 65536 available
Trap buffer: 2048 samples (4 tones x 512 samples each)
Total envelope samples (sweep + trap): 65536 / 65536 available
Running on hardware — 124 sweep steps x 48.39 us = 6.000 ms sweep, then trapping (4 tones, periodic, indefinitely until soc.reset_gens()).


In [6]:
soc.reset_gens()

In [11]:
# print(soccfg)
# print(soc)

QICK running on RFSoC4x2, software version 0.2.422

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 1024 decimated (1.852 us)
		triggered by output 7, pin 14, feedback to tProc input 0
		ADC tile 0, blk 0 is ADC_D
	1:	axis_readout_v